# 课时10：模块与包

| 主题     | 重点 | 难点 |
|--------|------|------|
| 模块（module） | 模块定义、五种导入方式、`__name__` 与 `__main__` | 模块缓存与 `sys.modules` |
| 包（package）    | 包结构、`__init__.py` 作用、`__all__`、相对导入 | 相对导入的包上下文限制 |
| 工程实践与进阶 | 模块搜索路径、命名空间包、循环导入排查、项目结构规范 | 循环导入的根因与解决方案 |

## 一、模块（module）

### 1. 模块定义
- 什么是模块？
    - 一个 `.py` 文件就是一个模块。文件名去掉 `.py` 后缀即为模块名。
- 为什么需要模块：
    - 代码复用：写一次，多处导入
    - 命名空间隔离：不同模块的同名变量不冲突
    - 可维护性：按职责拆分代码文件
- `__name__` 与 `__main__`
    - 每个模块都有一个 `__name__` 属性。直接运行时值为 '__main__'，被导入时值为模块名
    - 模块自测：在模块末尾添加 `if __name__ == "__main__":`下的代码只在直接运行时执行

### 2. 五种导入方式

In [1]:
# ---------- 五种导入方式 ----------
import mymodule                               # 方式 1：import 模块名
from mymodule import add, divide              # 方式 2：from 模块名 import 成员
import mymodule as calc                       # 方式 3：import 模块名 as 别名
from mymodule import add as plus              # 方式 3：from 模块名 import 成员 as 别名
from mymodule import *                        # 方式 5：from 模块名 import *
# 只有 __all__ 中列出的名字被导入，以下会报错，因为 VERSION 不在 __all__ 里：
# print(VERSION)   # NameError: name 'VERSION' is not defined


# ---------- 附加：查看模块信息 ----------
print("\n== 查看模块信息 ==")
print("模块名：", mymodule.__name__)
print("文件路径：", mymodule.__file__)
print("公开名字：", mymodule.__all__)
print("包含的成员：", [n for n in dir(mymodule) if not n.startswith("_")])


== 查看模块信息 ==
模块名： mymodule
文件路径： E:\Agent\llm_application_development_tutorial\01_Python_tutorial\03_代码工程化\mymodule.py
公开名字： ['add', 'divide', 'Calculator', 'PI']
包含的成员： ['Calculator', 'PI', 'VERSION', 'add', 'divide']


### 3. 模块导入只执行一次
- 模块只执行一次：重复 `import` 不会重新执行，直接从 `sys.modules` 取缓存
- 可以强制重载：`importlib.reload(module)`，主要用于开发调试

In [2]:
# Python 会缓存已导入的模块（sys.modules）
# 多次 import 同一个模块不会重复执行
import mymodule  # 第一次：执行模块代码
import mymodule  # 第二次：直接从缓存获取

# 强制重新加载
import importlib
importlib.reload(mymodule)

<module 'mymodule' from 'E:\\Agent\\llm_application_development_tutorial\\01_Python_tutorial\\03_代码工程化\\mymodule.py'>

### 4. 模块搜索路径
```
Python 按以下顺序查找模块：
1. 内置模块（sys.builtin_module_names）
2. sys.path 列表中的目录，按顺序查找：
   ├── 当前脚本所在目录（或交互环境的 cwd）
   ├── PYTHONPATH 环境变量中的路径
   ├── 标准库路径
   └── site-packages（第三方包安装目录）
可以动态添加搜索路径：`sys.path.append(path)`
```

In [3]:
import sys
for p in sys.path:
    print(p)

E:\Agent\llm_application_development_tutorial
E:\Agent\llm_application_development_tutorial\src
E:\Development_Tool\PyCharm 2025.2.4\plugins\python-ce\helpers\pydev
E:\Development_Tool\PyCharm 2025.2.4\plugins\python-ce\helpers\jupyter_debug
C:\Users\user\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\python312.zip
C:\Users\user\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\DLLs
C:\Users\user\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib
C:\Users\user\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none
E:\Agent\llm_application_development_tutorial\.venv

E:\Agent\llm_application_development_tutorial\.venv\Lib\site-packages


### 二、包（Package）


### 1. 包定义

- 什么是包？
    - 包是包含多个模块 module（及子包）的目录，用于组织相关代码
- 包的目录结构
```
├── main.py                 ← 包外调用方
└── mypackage/              ← 顶层包
    ├── __init__.py         ← 包标识文件（Python 3.3+ 可省略，但建议保留）
    ├── module_a.py         ← 子模块 A
    └── subpackage/         ← 子包
        ├── __init__.py
        └── module_b.py
```

### 2. `__init__.py`的作用
- 标记目录为包：Python 3.3 前必须存在；3.3+ 引入命名空间包后可省略，但保留更规范
- 包初始化代码：第一次 `import mypackage` 时执行，且只执行一次。
- `__all__`：控制 `from mypackage import *` 导入哪些名字。
- 接口提升：把子模块中的常用对象提到包级别,方便使用方:
    ```
       from mypackage import greet
   而不是
       from mypackage.module_a import greet
    ```
- 包级元数据：`__version__`、`__author__` 等。

①`__all__` 变量：
- 模块（module）或包（package）中定义的一个列表，控制 `from M import *` 时导出哪些名称
- 以 _ 开头的变量或函数，不在 `__all__` 中，不会被 import * 导出

②接口提升
```
# mypackage/__init__.py
from .module_a import greet

# 外部调用可以直接：from mypackage import greet
# 不需要写 from mypackage.module_a import greet
```

In [4]:
from mypackage import greet
greet("Alice")

[init mypackage] __name__=mypackage, __package__=mypackage, 已在sys.modules=True
[load module_a] __name__=mypackage.module_a, __package__=mypackage
[init subpackage] __name__=mypackage.subpackage, __package__=mypackage.subpackage
[load module_b] __name__=mypackage.subpackage.module_b, __package__=mypackage.subpackage


'Hello, Alice! (来自 mypackage.module_a)'

In [5]:
from mypackage.subpackage import module_b
module_b.add(1,3)

4

### 3. 相对导入与绝对导入

绝对导入（推荐）： 从项目根包开始，完整写出路径。

In [6]:
from mypackage.module_a import greet
from mypackage.subpackage.module_b import add

相对导入：从当前包开始，使用 `.` 表示当前包，`..` 表示上一级包。

```
mypackage/module_a.py
# 只能在包内使用:
#   from .      import xxx      # 当前包
#   from ..     import xxx      # 上一级包(父包)
#   from ..sub  import xxx      # 兄弟子包
#
# 直接用 `python module_a.py` 运行时,__package__ 为空,无法使用相对导入
# 相对导入会抛 ImportError: attempted relative import with no known parent package
```

## 三、易错点速查

| 序号 | 易错点 | 错因 | 纠正 |
|------|--------|------|------|
| 1 | 文件名与标准库重名 | 不知道搜索路径优先级 | 避免使用 `random.py`、`math.py` 等标准库同名 |
| 2 | 直接运行的脚本用相对导入 | `__main__` 没有包上下文 | 用绝对导入，或通过 `python -m` 运行 |
| 3 | `import *` 导致名称冲突 | 不知道 `__all__` 的作用 | 用显式导入或定义 `__all__` |
| 4 | 循环导入报错 | 模块间互相依赖 | 重构依赖关系或延迟导入 |
| 5 | 修改模块后导入不生效 | 不知道模块缓存机制 | 用 `importlib.reload()` 或重启解释器 |
| 6 | `__init__.py` 中写大量业务代码 | 不知道它的执行时机 | 只做重导出和初始化，业务逻辑放独立模块 |